# exostab quickstart

[exostab](https://github.com/mimitoz62/exostab) answers one question: is this known multi-planet system dynamically stable? Name a system, and it
tells you whether its planets are likely to keep their orbits, and how firmly the data support that answer. The
parameters come from the [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/), and every assumption about missing data is printed.

It asks in three independent ways, for the published values (the *nominal* parameters) and for 20 random draws within
the published error bars:

- **Hill spacing**: are neighbouring orbits far enough apart that the planets can never meet?
- **Angular momentum deficit (AMD)**: is there enough AMD for two neighbouring orbits to cross, or for the innermost
  planet to reach the star?
- **N-body integration with the MEGNO chaos indicator**: do the orbits stay regular when they are integrated forward?

Each indicator gives one of four verdicts:

- **stable**: its condition for stability holds for the nominal parameters and for at least 95% of the draws.
- **marginal**: borderline, or the draws disagree with the nominal parameters. This means mixed evidence, not "about
  to fall apart".
- **unstable**: it predicts instability for the nominal parameters and for at least 95% of the draws.
- **inconclusive**: it could not be evaluated reliably, for example because the energy error of the integration was
  too large.

The overall verdict is conservative. It is *stable* only if every indicator says stable, and *unstable* only if the
integration itself shows close encounters or escapes. It is *inconclusive* if the integrations could not be trusted,
and *marginal* in every other case, including any disagreement between the indicators. An N-body *stable* means that
no instability was found within the integrated time, not that the system is stable for billions of years.

**Run the cells in order.** In Colab, use **Runtime → Run all**. Colab asks you to sign in with a Google account, and to
confirm that you want to run a notebook from GitHub. The first code cell installs exostab and can take about a minute;
you can ignore its note about restarting the kernel. A check then takes from a few seconds to a few minutes, depending
on the number of planets and on the computer. The results are written to the folder `exostab-results/`. In Colab, the
folder icon on the left lists them for download; they are deleted when the session ends.

In [ ]:
# Installs exostab from GitHub. This can take about a minute.
%pip install --quiet "exostab @ git+https://github.com/mimitoz62/exostab@v1.0.1"

In [ ]:
import os

from IPython.display import Image, display

# Plain text in the notebook: no progress spinner, no colours and, in Colab, lines of up to 110 characters.
os.environ.update(TERM="dumb", COLUMNS="110", LINES="50")

!exostab --version

## 1. A check with the bundled snapshot

`--offline` uses the snapshot of 22 systems that ships with exostab (archive data of 28 September 2026), so this run can
be compared with the one in the [README](https://github.com/mimitoz62/exostab#quickstart). TRAPPIST-1 comes out **marginal**. Chaotic integrations are
sensitive to floating-point rounding, so the number of chaotic draws can differ by one or two on another computer.

In [ ]:
!exostab check "TRAPPIST-1" --offline

The report card lists the parameters used (a mark on every value that was assumed, derived, estimated or
randomized), one verdict per indicator with the draw counts, the overall verdict, and the data-completion rules that
were applied. Hill spacing and AMD are *sufficient* conditions or heuristics: failing them means that stability is
not guaranteed, not that the system falls apart. See [what the verdicts mean](https://mimitoz62.github.io/exostab/#what-the-verdicts-mean).

## 2. The figures

`orbits.png` shows the nominal orbits and the positions of the planets during the run. `megno.png` shows MEGNO ⟨Y⟩ and
the energy error of the nominal run (WHFast, checked with IAS15), and the final ⟨Y⟩ of every draw; ⟨Y⟩ ≥ 2.2 counts as
chaotic.

In [ ]:
display(Image("exostab-results/trappist-1/orbits.png", width=560))
display(Image("exostab-results/trappist-1/megno.png", width=900))

## 3. The results in Python

Every check writes `report.json`, which follows a versioned JSON schema that ships with the package. It holds the
verdicts, the numbers behind them, the value and source of every parameter, the completion rules, the settings and the
software versions.

In [ ]:
import json

with open("exostab-results/trappist-1/report.json") as f:
    report = json.load(f)

print("Overall verdict:", report["overall"]["verdict"])
for key, indicator in report["indicators"].items():
    d = indicator["draws"]
    print(
        f"  {key:6} {indicator['verdict']:9} draws: {d['stable']} stable, {d['marginal']} marginal, "
        f"{d['unstable']} unstable, {d['inconclusive']} inconclusive"
    )
print("Completion rules:", sorted({flag["rule"] for flag in report["system"]["flags"]}))
print("Archive data of:", report["data"]["retrieved_utc"], " seed:", report["settings"]["seed"])
print("Software:", report["software"])

## 4. Any system in the archive

Set `SYSTEM` to a host or planet name, then run this cell and the next one. Names are the archive's, as `exostab list`
below shows: Kepler-90, for example, is listed as KOI-351. The first check without `--offline` downloads the archive's
table of multi-planet systems (about 2,700 rows) and caches it, so this cell needs access to the internet. If the
archive cannot be reached, exostab uses the snapshot and says so. TOI-700 is not in the snapshot, so the check would
then stop with a message, and the next cell would have no figure to show.

In the archive table of 31 October 2026, none of the four planets of TOI-700 has a measured mass. The report card marks
the masses as estimated (ᵉ); the archive derived them from the planet radii.

In [ ]:
SYSTEM = "TOI-700"  # @param {type:"string"}
!rm -rf exostab-results/live
!exostab check "{SYSTEM}" --out exostab-results/live

In [ ]:
display(Image("exostab-results/live/orbits.png", width=560))

`exostab list` shows the systems in the archive table. Here, those with six or more planets.

In [ ]:
!exostab list --min-planets 6

## 5. What if?

`exostab sweep` varies one parameter of one planet and keeps the others at their nominal values. Here the eccentricity
of HD 127195 b takes 7 values from 0 to 0.6, with 2,000 orbits of the inner planet per value instead of the default
10,000 to keep it quick.

In [ ]:
!exostab sweep "HD 127195" --planet b --param e --range 0 0.6 --steps 7 --orbits 2000 --offline
display(Image("exostab-results/hd-127195/sweep.png", width=700))

AMD is the first indicator to change (marginal at e = 0.1). At e = 0.2 the integration is chaotic and Hill spacing
and AMD say unstable, and from e = 0.3 on every run ends in a close encounter or escape. Chaotic runs depend on
floating-point rounding, so a row near the transition can differ on another computer. `sweep.csv` has one row per value,
and `sweep.json` records the settings and the command.

## Next steps

- Look up any system in the weekly [stability catalog](https://mimitoz62.github.io/exostab/catalog/), which needs no installation.
- Install exostab on your own computer (Python 3.10 or newer, in a virtual environment):
  `pip install "exostab @ git+https://github.com/mimitoz62/exostab@v1.0.1"`.
- Check your own parameters with `exostab check --from-file my_system.yaml`, starting from
  [`examples/two_planet.yaml`](https://github.com/mimitoz62/exostab/blob/main/examples/two_planet.yaml).
- Use the [web app](https://mimitoz62.github.io/exostab/web/): the same analysis, with editable parameters and "what if" sliders.
- Read how the indicators work and what they cannot tell you: [Methods](https://mimitoz62.github.io/exostab/methods/),
  [Data and assumptions](https://mimitoz62.github.io/exostab/data-assumptions/) and [Limitations](https://github.com/mimitoz62/exostab#limitations).
- Every option of every command: [CLI reference](https://mimitoz62.github.io/exostab/cli/).
- If you use the results, please [cite exostab, the methods and the archive](https://mimitoz62.github.io/exostab/citing/).
- Questions about a system are welcome as [issues](https://github.com/mimitoz62/exostab/issues/new/choose).